In [1]:

import logging
import time
import sys
from typing import List, Optional

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('soilhive_automation.log')
    ]
)
logger = logging.getLogger(__name__)

In [2]:


class SoilHiveAutomator:
    def __init__(self, headless: bool = False, download_dir: str = None):
        """
        Initialize the SoilHive Automator.
        
        Args:
            headless (bool): Run browser in headless mode.
            download_dir (str, optional): Custom download directory.
        """
        self.headless = headless
        self.download_dir = download_dir
        self.driver = self.setup_driver()
        self.wait = WebDriverWait(self.driver, 20)
        self.base_url = "https://soilhive.ag/app/availability"

    def setup_driver(self) -> webdriver.Chrome:
        """
        Setup and configure the Chrome WebDriver.
        
        Returns:
            webdriver.Chrome: Configured Chrome driver instance.
        """
        logger.info("Setting up Chrome driver...")
        chrome_options = Options()
        
        if self.headless:
            chrome_options.add_argument("--headless=new")
            
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--start-maximized")
        
        # Disable automation flags to avoid detection (basic)
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option('useAutomationExtension', False)

        if self.download_dir:
            prefs = {"download.default_directory": self.download_dir}
            chrome_options.add_experimental_option("prefs", prefs)

        try:
            service = Service(ChromeDriverManager().install())
            driver = webdriver.Chrome(service=service, options=chrome_options)
            return driver
        except Exception as e:
            logger.error(f"Failed to setup driver: {e}")
            raise

    def close(self):
        """Close the browser driver."""
        if self.driver:
            logger.info("Closing driver...")
            self.driver.quit()

    def take_screenshot(self, name: str):
        """Take a screenshot for debugging purposes."""
        filename = f"screenshot_{name}_{int(time.time())}.png"
        self.driver.save_screenshot(filename)
        logger.info(f"Screenshot saved: {filename}")

    def load_page(self):
        """Load the SoilHive availability page and handle initial popups."""
        try:
            logger.info(f"Navigating to {self.base_url}")
            self.driver.get(self.base_url)
            
            # Handle Cookies if present
            try:
                cookie_btn = self.wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".coi-banner__accept")))
                cookie_btn.click()
                logger.info("Cookies accepted.")
            except TimeoutException:
                logger.info("No cookie banner found or already accepted.")

            # Click 'Start' button if present
            try:
                start_btn = self.wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Start')]")))
                start_btn.click()
                logger.info("Clicked 'Start' button.")
            except TimeoutException:
                 logger.warning("'Start' button not found, checking if already on map...")

        except Exception as e:
            logger.error(f"Error loading page: {e}")
            self.take_screenshot("load_page_error")
            raise

    def search_country(self, country_name: str):
        """
        Search for a country and select it from the dropdown.

        Args:
            country_name (str): Name of the country to search.
        """
        try:
            logger.info(f"Searching for country: {country_name}")
            
            # Wait for search input
            search_input = self.wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input#search")))
            search_input.clear()
            search_input.send_keys(country_name)
            
            # Wait for suggestions
            suggestion_locator = (By.CSS_SELECTOR, ".va-location__suggestions-item")
            self.wait.until(EC.presence_of_element_located(suggestion_locator))
            
            # Find the correct suggestion
            suggestions = self.driver.find_elements(*suggestion_locator)
            found = False
            for suggestion in suggestions:
                if country_name.lower() in suggestion.text.lower():
                    suggestion.click()
                    found = True
                    logger.info(f"Selected country: {country_name}")
                    break
            
            if not found:
                raise NoSuchElementException(f"Country '{country_name}' not found in suggestions.")

            # Wait for results to load (spinner or results container)
            time.sleep(2) # Give a moment for UI update
            
        except Exception as e:
            logger.error(f"Error searching country {country_name}: {e}")
            self.take_screenshot(f"search_error_{country_name}")
            raise

    def select_all_datasets(self):
        """Select all available datasets for the current location."""
        try:
            logger.info("Selecting all datasets...")
            select_all_label = self.wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".va-ds__select-all .va-checkbox")))
            select_all_label.click()
            logger.info("Clicked 'Select all'.")
            
        except Exception as e:
            logger.error(f"Error selecting datasets: {e}")
            self.take_screenshot("select_datasets_error")
            raise

    def click_download(self):
        """Click the download button to open the request form."""
        try:
            logger.info("Clicking Download button...")
            
            # Find the button by text using XPath as it's more robust for text content
            download_btn = self.wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Download')]")))
            
            # Scroll into view if needed
            self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", download_btn)
            time.sleep(1) # Stability
            
            download_btn.click()
            logger.info("Download button clicked.")
            
        except Exception as e:
            logger.error(f"Error clicking download: {e}")
            self.take_screenshot("click_download_error")
            raise

    def fill_email_and_submit(self, email: str):
        """
        Fill the email form and submit the request.

        Args:
            email (str): Email address to receive the data.
        """
        try:
            logger.info(f"Filling email: {email}")
            
            # Wait for email input in modal
            email_input = self.wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "input#email")))
            email_input.clear()
            email_input.send_keys(email)
            
            terms_checkbox = self.driver.find_element(By.CSS_SELECTOR, ".va-ds-download__content label.va-checkbox")
            self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", terms_checkbox)
            terms_checkbox.click()
            logger.info("Terms of Use accepted.")
            
            submit_btn = self.wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button.download-data-request-submit-button")))
            submit_btn.click()
            logger.info("Submit button clicked.")
            
            time.sleep(3) 
            logger.info("Request submitted successfully.")
            
        except Exception as e:
            logger.error(f"Error filling form: {e}")
            self.take_screenshot("form_error")
            raise

    def process_country(self, country: str, email: str):
        """
        Orchestrate the download process for a single country.
        """
        try:
            self.search_country(country)
            self.select_all_datasets()
            self.click_download()
            self.fill_email_and_submit(email)
            
            self.driver.refresh()
            self.wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input#search")))
            
        except Exception as e:
            logger.error(f"Failed to process {country}: {e}")
            # Try to recover by reloading page
            self.driver.refresh()


In [5]:

def main():
    # europa_countries = [
    #     "Albania", "Andorra", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina", 
    #     "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", 
    #     "Finland", "France", "Germany", "Greece", "Hungary", "Iceland", "Ireland", 
    #     "Italy", "Latvia", "Liechtenstein", "Lithuania", "Luxembourg", "Malta", 
    #     "Moldova", "Monaco", "Montenegro", "Netherlands", "North Macedonia", "Norway", 
    #     "Poland", "Portugal", "Romania", "Russia", "San Marino", "Serbia", "Slovakia", 
    #     "Slovenia", "Spain", "Sweden", "Switzerland", "Ukraine", "United Kingdom", "Vatican City"
    # ]

    africa_countries = [
        "Algeria", "Angola", "Benin", "Botswana", "Burkina Faso",
        "Burundi", "Cabo Verde", "Cameroon", "Central African Republic",
        "Chad", "Comoros", "Democratic Republic of the Congo",
        "Republic of the Congo", "Djibouti", "Egypt", "Equatorial Guinea",
        "Eritrea", "Eswatini", "Ethiopia", "Gabon", "Gambia", "Ghana",
        "Guinea", "Guinea-Bissau", "Ivory Coast", "Kenya", "Lesotho",
        "Liberia", "Libya", "Madagascar", "Malawi", "Mali",
        "Mauritania", "Mauritius", "Morocco", "Mozambique",
        "Namibia", "Niger", "Nigeria", "Rwanda",
        "Sao Tome and Principe", "Senegal", "Seychelles",
        "Sierra Leone", "Somalia", "South Africa",
        "South Sudan", "Sudan", "Tanzania", "Togo",
        "Tunisia", "Uganda", "Zambia", "Zimbabwe"
    ]
    usa_states = [
        "Alabama", "Alaska", "Arizona", "Arkansas", "California",
        "Colorado", "Connecticut", "Delaware", "Florida", "Georgia",
        "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
        "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland",
        "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri",
        "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey",
        "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
        "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina",
        "South Dakota", "Tennessee", "Texas", "Utah", "Vermont",
        "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"
    ]

    asia_countries = [
        "China", "India", "Indonesia", "Pakistan", "Bangladesh",
        "Vietnam", "Thailand", "Philippines", "Japan", "South Korea",
        "Myanmar", "Malaysia", "Nepal", "Sri Lanka",
        "Saudi Arabia", "Iran", "Iraq", "Turkey",
        "Kazakhstan", "Uzbekistan"
    ]

    # latin_america = [
    #     "Brazil", "Argentina", "Chile", "Colombia",
    #     "Peru", "Venezuela", "Ecuador", "Bolivia",
    #     "Paraguay", "Uruguay", "Mexico",
    #     "Guatemala", "Honduras", "Costa Rica"
    # ]


    countries =  asia_countries + usa_states + africa_countries
    
    email = "dsconite@gmail.com" 
    
    automator = SoilHiveAutomator(headless=True)
    
    try:
        automator.load_page()
        
        for country in countries:
            automator.process_country(country, email)
            
    except Exception as e:
        logger.critical(f"Critical error in main loop: {e}")
    finally:
        automator.close()

if __name__ == "__main__":
    main()



2026-02-19 20:22:49,652 - INFO - Setting up Chrome driver...
2026-02-19 20:22:49,664 - INFO - ====== WebDriver manager ======
2026-02-19 20:22:49,828 - INFO - Get LATEST chromedriver version for google-chrome
2026-02-19 20:22:50,157 - INFO - Get LATEST chromedriver version for google-chrome
2026-02-19 20:22:50,329 - INFO - Driver [/home/agbelgaid/.wdm/drivers/chromedriver/linux64/143.0.7499.192/chromedriver-linux64/chromedriver] found in cache
2026-02-19 20:22:51,119 - INFO - Navigating to https://soilhive.ag/app/availability
2026-02-19 20:22:55,238 - INFO - Cookies accepted.
2026-02-19 20:23:00,467 - INFO - Clicked 'Start' button.
2026-02-19 20:23:00,467 - INFO - Searching for country: China
2026-02-19 20:23:20,126 - INFO - Selected country: China
2026-02-19 20:23:22,127 - INFO - Selecting all datasets...
2026-02-19 20:23:23,406 - INFO - Clicked 'Select all'.
2026-02-19 20:23:23,406 - INFO - Clicking Download button...
2026-02-19 20:23:24,523 - INFO - Download button clicked.
2026-02-